<a href="https://colab.research.google.com/github/thanita47/Thai-Financial-Sentiment/blob/main/thai_financial_sentiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install datasets pandas

import pandas as pd
import requests

base_url = "https://raw.githubusercontent.com/PyThaiNLP/wisesight-sentiment/master/"

neg = requests.get(base_url + "neg.txt").text.strip().split("\n")
neu = requests.get(base_url + "neu.txt").text.strip().split("\n")
pos = requests.get(base_url + "pos.txt").text.strip().split("\n")

# DataFrame
df = pd.DataFrame(
    [(text, "neg") for text in neg] +
    [(text, "neu") for text in neu] +
    [(text, "pos") for text in pos],
    columns=["text", "label"]
)

# Cleaning
df["text"] = df["text"].str.replace("\r", "").str.strip()

print(df.shape)
print(df["label"].value_counts())
print(df.head())

(26162, 2)
label
neu    14561
neg     6823
pos     4778
Name: count, dtype: int64
  text label
0   ☹️   neg
1    😔   neg
2    😞   neg
3    😥   neg
4   รำ   neg


In [2]:
from sklearn.model_selection import train_test_split

df = df[df["text"] != "ERROR!"].reset_index(drop=True)

# แปลง label เป็นตัวเลข
label_map = {"neg": 0, "neu": 1, "pos": 2, "q": 3}
df["label_id"] = df["label"].map(label_map)

# แบ่ง train/valid/test
train_df, test_df = train_test_split(df, test_size=0.1, random_state=42)
train_df, valid_df = train_test_split(train_df, test_size=0.1, random_state=42)

print(train_df.shape, valid_df.shape, test_df.shape)
print(train_df["label"].value_counts())

(21190, 3) (2355, 3) (2617, 3)
label
neu    11783
neg     5531
pos     3876
Name: count, dtype: int64


In [3]:
!pip install transformers torch

from transformers import AutoTokenizer

model_name = "airesearch/wangchanberta-base-att-spm-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# ทดสอบ tokenizer
sample = train_df["text"].iloc[0]
print(sample)
print(tokenizer(sample))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/546 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/282 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/905k [00:00<?, ?B/s]

‡πÄ‡∏´‡∏≠‡∏∞‡πÜ‡πÜ ‡πÅ‡∏™‡∏á‡πÇ‡∏™‡∏°‡∏õ‡πà‡∏≤‡∏ß
{'input_ids': [5, 10, 3, 10, 3, 7066, 3, 6], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1]}


In [4]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from torch.utils.data import Dataset
import torch

# Dataset class
class ThaiSentimentDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128):
        self.texts = df["text"].tolist()
        self.labels = df["label_id"].tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }

# โหลด model
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=4)

# สร้าง dataset
train_dataset = ThaiSentimentDataset(train_df, tokenizer)
valid_dataset = ThaiSentimentDataset(valid_df, tokenizer)

print("Dataset ready:", len(train_dataset), len(valid_dataset))

model.safetensors:   0%|          | 0.00/423M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] CamembertForSequenceClassification LOAD REPORT from: airesearch/wangchanberta-base-att-spm-uncased
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Dataset ready: 21190 2355


In [5]:
from transformers import TrainingArguments, Trainer
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="weighted")
    }

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=100,
    fp16=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.154314,0.159803,0.949045,0.948864
2,0.104627,0.186227,0.954140,0.954019


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.154314,0.159803,0.949045,0.948864
2,0.104627,0.186227,0.954140,0.954019
3,0.051208,0.215004,0.954140,0.953948


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3975, training_loss=0.12740329034673342, metrics={'train_runtime': 536.2121, 'train_samples_per_second': 118.554, 'train_steps_per_second': 7.413, 'total_flos': 4181567535175680.0, 'train_loss': 0.12740329034673342, 'epoch': 3.0})

In [7]:
test_dataset = ThaiSentimentDataset(test_df, tokenizer)
results = trainer.evaluate(test_dataset)
print(results)

Training Loss,Validation Loss,Epoch,Accuracy,F1
0.051208,0.141326,3,0.953764,0.953561


{'eval_loss': 0.1413261741399765, 'eval_accuracy': 0.953763851738632, 'eval_f1': 0.9535612090213723}


In [8]:
def predict(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=128).to(model.device)
    outputs = model(**inputs)
    pred = outputs.logits.argmax(-1).item()
    label_map = {0: "neg", 1: "neu", 2: "pos", 3: "q"}
    return label_map[pred]

# ทดสอบ
print(predict("หุ้น PTT วันนี้ราคาพุ่งสูงขึ้นมาก นักลงทุนแห่ซื้อ"))
print(predict("ตลาดหุ้นไทยวันนี้ดิ่งหนัก นักลงทุนเทขาย"))
print(predict("ราคาน้ำมันวันนี้ทรงตัว"))

pos
neg
neg


In [12]:
!pip install feedparser yfinance

import feedparser
import yfinance as yf
import pandas as pd
from datetime import datetime

# ข่าวธุรกิจ/การเงิน
urls = [
    "https://www.prachachat.net/feed",
    "https://www.prachachat.net/finance/feed",
    "https://www.prachachat.net/stock-market/feed",
]

news_list = []
for url in urls:
    feed = feedparser.parse(url)
    print(f"{url} → {len(feed.entries)} entries")
    for entry in feed.entries:
        news_list.append({
            "title": entry.get("title", ""),
            "date": entry.get("published", ""),
            "source": url
        })

news_df = pd.DataFrame(news_list).drop_duplicates(subset="title").reset_index(drop=True)

# กรองเฉพาะข่าวการเงิน/หุ้น
finance_keywords = ["หุ้น", "ตลาด", "ลงทุน", "กำไร", "ขาดทุน", "เศรษฐกิจ",
                    "ธนาคาร", "ดอกเบี้ย", "งบ", "รายได้", "SET", "บริษัท", "ราคา", "ทุน"]

finance_df = news_df[news_df["title"].str.contains("|".join(finance_keywords), na=False)].reset_index(drop=True)

# Predict sentiment
finance_df["sentiment"] = finance_df["title"].apply(predict)

print(f"\nข่าวการเงินทั้งหมด: {len(finance_df)} ข่าว")
print(finance_df[["title", "sentiment"]].to_string())
print("\nสัดส่วน sentiment:")
print(finance_df["sentiment"].value_counts())

https://www.prachachat.net/feed → 30 entries
https://www.prachachat.net/finance/feed → 30 entries
https://www.prachachat.net/stock-market/feed → 0 entries

ข่าวการเงินทั้งหมด: 28 ข่าว
                                                                                                   title sentiment
0                                    AOT หนุนไทยขึ้นฮับเอวิเอชั่น ลงทุนเพิ่มคาพาซิตี้รับ 160 ล้านคนต่อปี       pos
1                                                                                            ไวกิ้งบุก !       pos
2          ‘พลพีร์’ สั่งสแกนทั้งเกาะกว่า 10,000 บริษัท ตรวจสอบมาเฟียภูเก็ต ชี้มีนอมินีมากกว่า 400 บริษัท       neg
3   พ่อค้า-แม่ค้า ‘ตลาดนัดจตุจักร’ ฝากผู้ว่าฯ กทม.คนใหม่ ดูแลทางเท้า-ค่าเช่า ‘เจ้าสัวใบหยก’ จี้แก้ทุจริต       neg
4           ‘สิริพงศ์’ เร่งทางคู่ ผุดถนนสายใหม่-ศูนย์ขนส่งชายแดนนครพนม หนุนเศรษฐกิจ-ท่องเที่ยว “ไทย-ลาว”       pos
5                                                      คุยกับมือปั้น ‘Dime!’ ถอดรหัสแอปลงทุนขวัญใจ Gen Z       pos
6          

In [14]:
import yfinance as yf
from datetime import datetime, timedelta

# ดึงราคาหุ้น SET index
set_index = yf.download("^SET.BK", start="2026-06-01", end="2026-06-28")
print(set_index.tail(10))

/tmp/ipykernel_3430/1918515182.py:5: FutureWarning: YF.download() has changed argument auto_adjust default to True
  set_index = yf.download("^SET.BK", start="2026-06-01", end="2026-06-28")
[*********************100%***********************]  1 of 1 completed

Price             Close         High          Low         Open   Volume
Ticker          ^SET.BK      ^SET.BK      ^SET.BK      ^SET.BK  ^SET.BK
Date                                                                   
2026-06-15  1591.719971  1609.709961  1585.609985  1607.890015  9058700
2026-06-16  1588.050049  1596.229980  1583.959961  1595.640015  3951600
2026-06-17  1587.069946  1590.040039  1574.140015  1586.579956  4751200
2026-06-18  1585.060059  1596.439941  1583.239990  1587.829956  5005700
2026-06-19  1572.500000  1592.339966  1570.099976  1591.719971  3836300
2026-06-22  1574.130005  1580.859985  1568.050049  1579.109985  3155300
2026-06-23  1540.900024  1569.780029  1538.689941  1568.189941  5007800
2026-06-24  1548.219971  1553.349976  1536.369995  1547.729980  3063200
2026-06-25  1558.550049  1567.469971  1554.510010  1558.530029  5322400
2026-06-26  1542.339966  1549.819946  1537.959961  1548.270020  3838200


In [15]:
import pandas as pd
from datetime import datetime

# แปลง date ของข่าวให้เป็น date เดียวกัน
finance_df["date_only"] = pd.to_datetime(finance_df["date"], utc=True).dt.date

# แปลง SET index
set_df = set_index.reset_index()
set_df.columns = ["date", "close", "high", "low", "open", "volume"]
set_df["date"] = pd.to_datetime(set_df["date"]).dt.date

# คำนวณ % change ของราคาหุ้นวันถัดไป
set_df["next_close"] = set_df["close"].shift(-1)
set_df["price_change"] = ((set_df["next_close"] - set_df["close"]) / set_df["close"]) * 100

# รวม sentiment รายวัน
daily_sentiment = finance_df.groupby("date_only")["sentiment"].apply(
    lambda x: (x == "pos").sum() - (x == "neg").sum()
).reset_index()
daily_sentiment.columns = ["date", "sentiment_score"]

# จับคู่
merged = pd.merge(daily_sentiment, set_df[["date", "price_change"]], on="date", how="inner")
print(merged)

         date  sentiment_score  price_change
0  2026-06-25                1     -1.040075
1  2026-06-26               -3           NaN


In [16]:
!pip install shap

import shap
import torch
import numpy as np

# สร้าง explainer
def predict_proba(texts):
    inputs = tokenizer(list(texts), return_tensors="pt",
                      truncation=True, max_length=128,
                      padding=True).to(model.device)
    with torch.no_grad():
        outputs = model(**inputs)
    probs = torch.softmax(outputs.logits, dim=-1).cpu().numpy()
    return probs

# เลือกข่าวมาอธิบาย
sample_texts = finance_df["title"].tolist()[:10]

explainer = shap.Explainer(predict_proba, shap.maskers.Text(tokenizer))
shap_values = explainer(sample_texts)

# plot
shap.plots.text(shap_values[0])

PartitionExplainer explainer: 11it [00:19,  3.17s/it]


In [17]:
print("=== PROJECT SUMMARY ===")
print(f"Model: WangchanBERTa fine-tuned on Wisesight Sentiment")
print(f"Train size: {len(train_df)} | Valid: {len(valid_df)} | Test: {len(test_df)}")
print(f"Test Accuracy: 95.38% | F1: 95.36%")
print(f"\nFinancial News Analyzed: {len(finance_df)} articles")
print(f"Sentiment Distribution:\n{finance_df['sentiment'].value_counts()}")

=== PROJECT SUMMARY ===
Model: WangchanBERTa fine-tuned on Wisesight Sentiment
Train size: 21190 | Valid: 2355 | Test: 2617
Test Accuracy: 95.38% | F1: 95.36%

Financial News Analyzed: 28 articles
Sentiment Distribution:
sentiment
pos    14
neg    14
Name: count, dtype: int64
